# **Introdução**

### Descrição do projeto

Você trabalha para a loja online Ice, que vende videogames no mundo todo. As avaliações de usuários e especialistas, gêneros, plataformas e dados históricos sobre vendas de jogos estão disponíveis em fontes abertas. 

Os dados disponibilizados remontam a 2016. Vamos imaginar que estamos em dezembro de 2016 e você está planejando uma campanha para 2017.

### Descrição dos dados

O dataset compactado `games.csv` possui os seguintes atributos/colunas:

* `Name` (Nome do produto)
* `Platform` (Plataforma, como Xbox ou PlayStation)
* `Year_of_Release` (Ano de lançamento)
* `Genre` (Gênero, como Esporte ou Corrida)
* `NA_sales` (Vendas norte-americanas em milhões de USD)
* `EU_sales` (Vendas na Europa em milhões de USD)
* `JP_sales` (Vendas no Japão em milhões de USD)
* `Other_sales` (Vendas em outros países em milhões de USD)
* `Critic_Score` (Pontuação crítica, máximo de 100)
* `User_Score` (Pontuação dos usuários, máximo de 10)
* `Classificação` (ESRB classificação etária, como Teen ou Mature)

### Objetivo

Identificar padrões que determinam se um jogo tem sucesso ou não. Isso vai permitir que você identifique possíveis sucessos e planeje campanhas publicitárias.

# **Setup e Dados**

## Ambiente

### Importação bibliotecas

In [1]:
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import seaborn as sns
from scipy import stats as st

### Configurações Globais

In [2]:
# reprodutibilidade — seed único
RANDOM_STATE = 42

# exibição de dados
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# estilo visual
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

### Carregamento de dados

In [3]:
# leitura arquivo CSV
games_raw = pd.read_csv('../data/raw/games.csv')

## Pré Processamento

### Primeiras impressões

In [4]:
games_raw.head()

,Name,Platform,Year_of_Release,Genre,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score,User_Score,Rating
0,Wii Sports,Wii,2006.00,Sports,41.36,28.96,3.77,8.45,76.00,8,E
1,Super Mario Bros.,NES,1985.00,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.00,Racing,15.68,12.76,3.79,3.29,82.00,8.3,E
3,Wii Sports Resort,Wii,2009.00,Sports,15.61,10.93,3.28,2.95,80.00,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.00,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [5]:
games_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 16715 entries, 0 to 16714
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16713 non-null  str    
 1   Platform         16715 non-null  str    
 2   Year_of_Release  16446 non-null  float64
 3   Genre            16713 non-null  str    
 4   NA_sales         16715 non-null  float64
 5   EU_sales         16715 non-null  float64
 6   JP_sales         16715 non-null  float64
 7   Other_sales      16715 non-null  float64
 8   Critic_Score     8137 non-null   float64
 9   User_Score       10014 non-null  str    
 10  Rating           9949 non-null   str    
dtypes: float64(6), str(5)
memory usage: 1.4 MB


O dataset `games_raw` não possui a nomenclatura de suas colunas dentro do padrão *snake case*.

Seria mais adequado mudanças nos tipos de dados em `Year_of_Release` e `User_Score` para `int`, visando operações temporais e numéricas. 

É perceptível a presença de valores nulos nos atributos: `Name` (2), `Year_of_Release` (269), `Genre` (2), `Critic_Score`(8578), `User_Score` (6701) e `Rating` (6766). 
Cada atributo será tratado individualmente, com estratégia adequada ao seu tipo e volume de ausência.


### Tratamento de valores duplicados

In [6]:
# criação de cópia
games = games_raw.copy()

In [7]:
# descoberta quantidade de registros duplicados
print(f'A quantidade de registros duplicados presentes no dataset Games é: {games.duplicated().sum()}')

A quantidade de registros duplicados presentes no dataset Games é: 0


Não foram detectados registros completamente duplicados no dataset. 

Registros com mesmo `Name` e `Platform` não são considerados duplicatas — um mesmo jogo pode ter sido relançado na mesma plataforma em edições distintas, o que justifica a coexistência de entradas idênticas nesses dois atributos.

### Tratamento de valores nulos 

#### Atributo `Name` e `Genre`

In [8]:
# visulização dataset filtrado por nulos em Name
games[games['Name'].isnull()]

,Name,Platform,Year_of_Release,Genre,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score,User_Score,Rating
659,NaN,GEN,1993.00,NaN,1.78,0.53,0.00,0.08,NaN,NaN,NaN
14244,NaN,GEN,1993.00,NaN,0.00,0.00,0.03,0.00,NaN,NaN,NaN


In [ ]:
# preenchimento com designação neutra
games['Name'] = games['Name'].fillna('unknown')
games['Genre'] = games['Genre'].fillna('unknown')

Os 2 registros sem `Name` e `Genre` (0.012% do dataset) correspondem a entradas da plataforma GEN datadas de 1993, sem qualquer identificação recuperável. 

A remoção desses registros foi descartada por serem portadores de dados válidos de vendas. A designação `unknown` foi adotada como marcação neutra que não interfere em agrupamentos nem introduz viés nas análises.

#### Atributo `Year_of_Release` 

In [10]:
# visulização dataset filtrado por nulos em Year_of_release
games[games['Year_of_Release'].isnull()]

,Name,Platform,Year_of_Release,Genre,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score,User_Score,Rating
183,Madden NFL 2004,PS2,NaN,Sports,4.26,0.26,0.01,0.71,94.00,8.5,E
377,FIFA Soccer 2004,PS2,NaN,Sports,0.59,2.36,0.04,0.51,84.00,6.4,E
456,LEGO Batman: The Videogame,Wii,NaN,Action,1.80,0.97,0.00,0.29,74.00,7.9,E10+
475,wwe Smackdown vs. Raw 2006,PS2,NaN,Fighting,1.57,1.02,0.00,0.41,NaN,NaN,NaN
609,Space Invaders,2600,NaN,Shooter,2.36,0.14,0.00,0.03,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
16373,PDC World Championship Darts 2008,PSP,NaN,Sports,0.01,0.00,0.00,0.00,43.00,tbd,E10+
16405,Freaky Flyers,GC,NaN,Racing,0.01,0.00,0.00,0.00,69.00,6.5,T
16448,Inversion,PC,NaN,Shooter,0.01,0.00,0.00,0.00,59.00,6.7,M
16458,Hakuouki: Shinsengumi Kitan,PS3,NaN,Adventure,0.01,0.00,0.00,0.00,NaN,NaN,NaN


In [ ]:
# criação de função para preenchimento de nulos em Year_of_Release com base na mediana dos registros do mesmo jogo em outras plataformas, ou mediana do genêro em plataforma  
def fixing_year(row):
    name = row['Name']
    year = row['Year_of_Release']
    platform = row['Platform']
    genre = row['Genre']

    # teste valor nulo em atributo ano
    if pd.notna(year):
        return year

    match = games[(games['Name'] == name) & (games['Platform'] != platform)]['Year_of_Release']
    median_year = match.median()
    
    # teste valor nulo após filtro por jogo
    if pd.notna(median_year):
        return median_year

    match = games[(games['Platform'] == platform) & (games['Genre'] == genre)]['Year_of_Release']
    median_year = match.median()

    # teste valor nulo após filtro por plataforma e genêro
    if pd.notna(median_year):
        return median_year

    return year

In [14]:
# aplicação da função para preenchimento de nulo em Year_of_Release
games['Year_of_Release'] = games.apply(fixing_year, axis=1) 

In [15]:
# quantidade de valores nulos em Year_of_Release após uso de função
games['Year_of_Release'].isnull().sum()

np.int64(0)

A função `fixing_year` aplica uma lógica de preenchimento em dois níveis hierárquicos:

1. **Mesmo jogo, outras plataformas:** se o título existe em outra plataforma com ano conhecido, usa-se a mediana desses registros — pressuposto de que um jogo tende a ser lançado em múltiplas plataformas no mesmo período.
2. **Mesma plataforma e gênero:** como fallback, utiliza-se a mediana dos jogos do mesmo gênero na mesma plataforma — aproximação razoável para estimar a janela temporal de lançamentos similares.

Os 269 registros ausentes (~1.6% do dataset) foram todos preenchidos com base em outros registros do mesmo jogo, ou como plano B: com base em registros sobre o genêro e plataforma.  

#### Atributo `Critic_Score`

In [16]:
# visulização dataset filtrado por nulos em Critic_Score
games[games['Critic_Score'].isnull()]

,Name,Platform,Year_of_Release,Genre,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score,User_Score,Rating
1,Super Mario Bros.,NES,1985.00,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
4,Pokemon Red/Pokemon Blue,GB,1996.00,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN
5,Tetris,GB,1989.00,Puzzle,23.20,2.26,4.22,0.58,NaN,NaN,NaN
9,Duck Hunt,NES,1984.00,Shooter,26.93,0.63,0.28,0.47,NaN,NaN,NaN
10,Nintendogs,DS,2005.00,Simulation,9.05,10.95,1.93,2.74,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
16710,Samurai Warriors: Sanada Maru,PS3,2016.00,Action,0.00,0.00,0.01,0.00,NaN,NaN,NaN
16711,LMA Manager 2007,X360,2006.00,Sports,0.00,0.01,0.00,0.00,NaN,NaN,NaN
16712,Haitaka no Psychedelica,PSV,2016.00,Adventure,0.00,0.00,0.01,0.00,NaN,NaN,NaN
16713,Spirits & Spells,GBA,2003.00,Platform,0.01,0.00,0.00,0.00,NaN,NaN,NaN


In [17]:
# criação de função para preenchimento de nulos em Critic_Score com base na mediana dos registros do mesmo jogo em outras plataformas, ou com base na mediana do genêro em plataforma  
def fixing_critic(row):
    name = row['Name']
    critic = row['Critic_Score']
    platform = row['Platform']
    genre = row['Genre']

    # teste valor nulo em atributo critic_score
    if pd.notna(critic):
        return critic

    match = games[(games['Name'] == name) & (games['Platform'] != platform)]['Critic_Score']
    median_critic = match.median()
    
    # teste valor nulo após filtro por jogo
    if pd.notna(median_critic):
        return median_critic

    match = games[(games['Platform'] == platform) & (games['Genre'] == genre)]['Critic_Score']
    median_critic = match.median()

    # teste valor nulo após filtro por plataforma e genêro
    if pd.notna(median_critic):
        return median_critic

    return critic

In [18]:
# aplicação da função para preenchimento de nulo em Critic_Score
games['Critic_Score'] = games.apply(fixing_critic, axis=1) 

In [19]:
# quantidade de valores nulos em Critic_Score após uso de função
games['Critic_Score'].isnull().sum()

np.int64(1074)

A função `fixing_critic` segue a mesma hierarquia de `fixing_year`: primeiro busca o mesmo jogo em outras plataformas; na ausência de correspondência, utiliza a mediana dos jogos do mesmo gênero na plataforma. 

O volume inicial de ausentes em `Critic_Score` é expressivo (~51.3% do dataset). Dos 8.578 registros ausentes, 7.504 (~87.5% destes) foram preenchidos por imputação. 

Os 1.074 restantes (~6.4% do dataset) não possuem nem referência por título nem por gênero+plataforma. 

Esses registros são mantidos: removê-los excluiria jogos com dados de vendas válidos e introduziria viés de cobertura — jogos avaliados pela crítica tendem a ser os de maior visibilidade.

#### Atributo `User_Score`

In [20]:
# visulização dataset filtrado por nulos em User_Score
games[games['User_Score'].isnull()]

,Name,Platform,Year_of_Release,Genre,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score,User_Score,Rating
1,Super Mario Bros.,NES,1985.00,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
4,Pokemon Red/Pokemon Blue,GB,1996.00,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN
5,Tetris,GB,1989.00,Puzzle,23.20,2.26,4.22,0.58,NaN,NaN,NaN
9,Duck Hunt,NES,1984.00,Shooter,26.93,0.63,0.28,0.47,NaN,NaN,NaN
10,Nintendogs,DS,2005.00,Simulation,9.05,10.95,1.93,2.74,65.50,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
16710,Samurai Warriors: Sanada Maru,PS3,2016.00,Action,0.00,0.00,0.01,0.00,71.00,NaN,NaN
16711,LMA Manager 2007,X360,2006.00,Sports,0.00,0.01,0.00,0.00,73.00,NaN,NaN
16712,Haitaka no Psychedelica,PSV,2016.00,Adventure,0.00,0.00,0.01,0.00,72.00,NaN,NaN
16713,Spirits & Spells,GBA,2003.00,Platform,0.01,0.00,0.00,0.00,71.00,NaN,NaN


In [21]:
# criação de função para preenchimento de nulos em User_Score com base na mediana dos registros do mesmo jogo em outras plataformas, ou com base na mediana do genêro em plataforma  
def fixing_user(row):
    name = row['Name']
    user = row['User_Score']
    platform = row['Platform']
    genre = row['Genre']

    # teste valor nulo em atributo user_score
    if pd.notna(user):
        return user

    match = games[(games['Name'] == name) & (games['Platform'] != platform)]['User_Score']
    match = pd.to_numeric(match, errors='coerce')
    median_user = match.median()
    
    # teste valor nulo após filtro por jogo
    if pd.notna(median_user):
        return median_user

    match = games[(games['Platform'] == platform) & (games['Genre'] == genre)]['User_Score']
    match = pd.to_numeric(match, errors='coerce')
    median_user = match.median()

    # teste valor nulo após filtro por plataforma e genêro
    if pd.notna(median_user):
        return median_user

    return user

In [22]:
# aplicação da função para preenchimento de nulo em User_Score
games['User_Score'] = games.apply(fixing_user, axis=1) 

In [23]:
# quantidade de valores nulos em User_Score após uso de função
games['User_Score'].isnull().sum()

np.int64(1073)

A função `fixing_user` aplica a mesma hierarquia das anteriores. 

`User_Score` apresenta uma particularidade: o dado original contém entradas `'tbd'` (*to be determined*), convertidas para `NaN` via `pd.to_numeric(errors='coerce')` antes do cálculo da mediana — necessário para evitar que strings contaminem a agregação numérica.

Dos 6.701 ausentes (~40.1% do dataset), 5.628 (~84% destes) foram preenchidos. 

Os 1.073 remanescentes (~6.4% do dataset) são mantidos pelo mesmo critério aplicado a `Critic_Score`: não há base de imputação válida dentro do próprio dataset, e a remoção enviesaria a amostra contra jogos sem avaliação de usuários.

#### Atributo `Rating`

In [24]:
# visulização dataset filtrado por nulos em Rating
games[games['Rating'].isnull()]

,Name,Platform,Year_of_Release,Genre,NA_sales,EU_sales,JP_sales,Other_sales,Critic_Score,User_Score,Rating
1,Super Mario Bros.,NES,1985.00,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
4,Pokemon Red/Pokemon Blue,GB,1996.00,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN
5,Tetris,GB,1989.00,Puzzle,23.20,2.26,4.22,0.58,NaN,NaN,NaN
9,Duck Hunt,NES,1984.00,Shooter,26.93,0.63,0.28,0.47,NaN,NaN,NaN
10,Nintendogs,DS,2005.00,Simulation,9.05,10.95,1.93,2.74,65.50,7.30,NaN
...,...,...,...,...,...,...,...,...,...,...,...
16710,Samurai Warriors: Sanada Maru,PS3,2016.00,Action,0.00,0.00,0.01,0.00,71.00,7.20,NaN
16711,LMA Manager 2007,X360,2006.00,Sports,0.00,0.01,0.00,0.00,73.00,6.60,NaN
16712,Haitaka no Psychedelica,PSV,2016.00,Adventure,0.00,0.00,0.01,0.00,72.00,7.60,NaN
16713,Spirits & Spells,GBA,2003.00,Platform,0.01,0.00,0.00,0.00,71.00,8.05,NaN


In [25]:
# criação de função para preenchimento de nulos em Rating com base na mediana dos registros do mesmo jogo em outras plataformas, ou com base na mediana do genêro em plataforma  
def fixing_rating(row):
    name = row['Name']
    rating = row['Rating']
    platform = row['Platform']
    genre = row['Genre']

    # teste valor nulo em atributo user_score
    if pd.notna(rating):
        return rating

    match = games[(games['Name'] == name) & (games['Platform'] != platform)]['Rating']
    mode_rating = match.mode()
    
    # teste valor nulo após filtro por jogo
    if not mode_rating.empty:
        return mode_rating.iloc[0]

    match = games[(games['Platform'] == platform) & (games['Genre'] == genre)]['Rating']
    mode_rating = match.mode()

    # teste valor nulo após filtro por plataforma e genêro
    if not mode_rating.empty:
        return mode_rating.iloc[0]
    
    return rating

In [26]:
# aplicação da função para preenchimento de nulo em Rating
games['Rating'] = games.apply(fixing_rating, axis=1) 

In [27]:
# quantidade de valores nulos em Rating após uso de função
games['Rating'].isnull().sum()

np.int64(1066)

`Rating` é uma variável categórica (classificação ESRB: `EC`, `E`, `E10+`, `T`, `M`, `AO`). Por isso, o preenchimento utiliza **moda** — o valor mais frequente no grupo de referência — em vez de mediana, que seria semanticamente inválida para categorias. A mesma hierarquia é mantida: mesmo jogo em outras plataformas; como fallback, mesma plataforma e gênero.

Dos 6.766 ausentes (~40.5% do dataset), 5.700 (~84.3% destes) foram preenchidos. 

Os 1.066 remanescentes (~6.4% do dataset) são mantidos — `Rating` será usada como variável de segmentação auxiliar, e sua ausência não compromete as análises centrais de vendas.

### Nomenclatura colunas

In [29]:
# nomenclatura de colunas
to_fix_columns = games.columns

In [30]:
# função para adoção do método snake_case em colunas
def rename_columns(columns):
    fixed_columns = list()
    for column in columns:
        column = column.strip().lower().replace(' ', '_')
        fixed_columns.append(column)
    return fixed_columns

In [ ]:
# aplicação da função rename_columns
games.columns = rename_columns(to_fix_columns)

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='str')

A padronização para *snake_case* garante consistência no acesso a colunas por índice string ao longo de todo o notebook, eliminando necessidade de ajustes pontuais por capitalização.

### Alteração tipo de dado

In [ ]:
# transformação year_of_release 
games['year_of_release'] = games['year_of_release'].astype(int)

In [37]:
# transformação user_score
games['user_score'] = pd.to_numeric(games['user_score'], errors='coerce')

A coluna `year_of_release` foi convertido para `int` — o atributo representa o ano calendário. Será suficiente para preservar a compatibilidade com agrupamentos e filtros de intervalo usados nas análises.

A coluna `user_score` foi convertido para de `str` para `float64` após conversão de entradas `'tbd'` (*to be determined*) para `NaN` — strings que bloqueiam operações numéricas. 

### Engenharia de Características

In [41]:
# criação coluna total_sales
games['total_sales'] = games['na_sales'] + games['eu_sales'] + games['jp_sales'] + games['other_sales']

`total_sales` agrega as vendas das quatro regiões disponíveis (`na_sales`, `eu_sales`, `jp_sales`, `other_sales`), representando o volume global de vendas de cada registro. 

Essa coluna será a principal métrica de desempenho comercial nas análises seguintes.

# **EDA**